In [ ]:
# ============================================================
# A02 — STEP 1
# COLAB + GOOGLE DRIVE RESTORE/AUDIT
#
# NO TRAINING
# NO OUTER EVALUATION
# ============================================================

from google.colab import drive
from pathlib import Path
import tensorflow as tf
import json

# ------------------------------------------------------------
# 1. Mount persistent Google Drive
# ------------------------------------------------------------

drive.mount(
    "/content/drive",
    force_remount=True
)

print("\nDrive mount: PASS")


# ------------------------------------------------------------
# 2. GPU audit
# ------------------------------------------------------------

print("\nTensorFlow:", tf.__version__)

gpus = tf.config.list_physical_devices(
    "GPU"
)

print("GPU:", gpus)

assert gpus, (
    "No GPU detected. "
    "Go to Runtime -> Change runtime type -> GPU."
)


# ------------------------------------------------------------
# 3. Locate the original project
# ------------------------------------------------------------

PROJECT_CANDIDATES = [

    Path(
        "/content/drive/"
        ".shortcut-targets-by-id/"
        "1un2SaLv7_DvXerUlPtzKSNZ_j-mu1qFQ/"
        "EEG_GANet_Reproduction"
    ),

    Path(
        "/content/drive/MyDrive/"
        "EEG_GANet_Reproduction"
    ),
]


PROJECT_ROOT = None

for candidate in PROJECT_CANDIDATES:

    print(
        "Project candidate:",
        candidate,
        "->",
        "FOUND"
        if candidate.is_dir()
        else "missing"
    )

    if (
        PROJECT_ROOT is None
        and
        candidate.is_dir()
    ):

        PROJECT_ROOT = candidate


assert PROJECT_ROOT is not None, (
    "Could not find EEG_GANet_Reproduction "
    "in this Google Drive account."
)


print(
    "\nPROJECT_ROOT:"
)

print(
    PROJECT_ROOT
)


# ------------------------------------------------------------
# 4. Locate A02 persistent recovery root
# ------------------------------------------------------------

RECOVERY_CANDIDATES = [

    Path(
        "/content/drive/MyDrive/"
        "A02_confirmatory_recovery/"
        "paper_confirmatory_cv_v1"
    ),

    (
        PROJECT_ROOT
        / "GLASS_GANet"
        / "results"
        / "paper_confirmatory_cv_v1"
    ),
]


A02_PERSISTENT_ROOT = None


for candidate in RECOVERY_CANDIDATES:

    a02_dir = (
        candidate
        / "subject_runs"
        / "A02"
    )

    print(
        "Recovery candidate:",
        candidate,
        "->",
        "FOUND"
        if a02_dir.is_dir()
        else "missing"
    )

    if (
        A02_PERSISTENT_ROOT is None
        and
        a02_dir.is_dir()
    ):

        A02_PERSISTENT_ROOT = (
            candidate
        )


assert A02_PERSISTENT_ROOT is not None, (
    "\nA02 recovery directory was not found.\n"
    "DO NOT TRAIN ANYTHING."
)


print(
    "\nA02_PERSISTENT_ROOT:"
)

print(
    A02_PERSISTENT_ROOT
)


# ------------------------------------------------------------
# 5. Prepared A02 data
# ------------------------------------------------------------

PREPARED_CANDIDATES = [

    (
        PROJECT_ROOT
        / "GLASS_GANet"
        / "prepared_data"
        / "A02_GLASS_preprocessed_seed42.npz"
    ),

    (
        A02_PERSISTENT_ROOT
        / "prepared_data"
        / "A02_GLASS_preprocessed_seed42.npz"
    ),
]


A02_PREPARED = None


for candidate in PREPARED_CANDIDATES:

    print(
        "Prepared-data candidate:",
        candidate,
        "->",
        "FOUND"
        if candidate.is_file()
        else "missing"
    )

    if (
        A02_PREPARED is None
        and
        candidate.is_file()
    ):

        A02_PREPARED = candidate


# Do not fail yet if prepared NPZ is absent;
# we can restore it from the frozen preprocessing later.

print(
    "\nPrepared A02:",
    A02_PREPARED
)


# ------------------------------------------------------------
# 6. Audit saved training progress
# ------------------------------------------------------------

A02_RUN_ROOT = (
    A02_PERSISTENT_ROOT
    / "subject_runs"
    / "A02"
)


MODEL_NAMES = [

    "structured_dbnet",

    "glass_gated_dbnet",

    "interaction_dbnet",

    "binary_dbnet",
]


print(
    "\n"
    + "=" * 100
)

print(
    "A02 SAVED TRAINING PROGRESS"
)

print(
    "=" * 100
)


for fold in range(
    1,
    8
):

    fold_dir = (
        A02_RUN_ROOT
        / f"outer_fold_{fold}"
    )


    stage1 = (
        fold_dir
        / "glass"
        / "stage1_no_shrinkage.npz"
    )


    stage2 = (
        fold_dir
        / "glass"
        / "stage2_shrinkage.npz"
    )


    checkpoint_count = 0
    summary_count = 0
    completed_pairs = []


    for seed in [
        1,
        2,
        3
    ]:

        for model_name in MODEL_NAMES:

            run_dir = (
                fold_dir
                / f"seed_{seed}"
                / model_name
            )


            weight_file = (
                run_dir
                / "best.weights.h5"
            )


            summary_file = (
                run_dir
                / "training_summary.json"
            )


            if weight_file.is_file():

                checkpoint_count += 1


            if summary_file.is_file():

                summary_count += 1


            if (
                weight_file.is_file()
                and
                summary_file.is_file()
            ):

                try:

                    record = json.loads(
                        summary_file.read_text(
                            encoding="utf-8"
                        )
                    )


                    if (
                        record.get(
                            "status"
                        )
                        ==
                        "completed"
                    ):

                        completed_pairs.append(
                            (
                                seed,
                                model_name
                            )
                        )

                except Exception:

                    pass


    marker = (
        fold_dir
        / "fold_training_complete.json"
    )


    print(
        f"\nFold {fold}"
    )

    print(
        "  directory:    ",
        "YES"
        if fold_dir.is_dir()
        else "NO"
    )

    print(
        "  GLASS Stage1: ",
        "PASS"
        if stage1.is_file()
        else "NO"
    )

    print(
        "  GLASS Stage2: ",
        "PASS"
        if stage2.is_file()
        else "NO"
    )

    print(
        f"  checkpoints:  "
        f"{checkpoint_count}/12"
    )

    print(
        f"  summaries:    "
        f"{summary_count}/12"
    )

    print(
        f"  completed:    "
        f"{len(completed_pairs)}/12"
    )

    print(
        "  fold marker:  ",
        "PASS"
        if marker.is_file()
        else "NO"
    )


    if completed_pairs:

        print(
            "  reusable models:"
        )

        for (
            seed,
            model_name
        ) in completed_pairs:

            print(
                f"    seed {seed} | "
                f"{model_name}"
            )


# ------------------------------------------------------------
# 7. Final status
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 100
)

print(
    "A02 NEW-DEVICE DRIVE AUDIT COMPLETE"
)

print(
    "=" * 100
)

print(
    "\nTraining performed: NO"
)

print(
    "Outer evaluation performed: NO"
)

print(
    "\nDO NOT DELETE OR MODIFY ANY "
    "EXISTING A02 FOLD DIRECTORIES."
)

In [ ]:
# ============================================================
# A02 — STEP 2
# SEARCH ALL LIKELY DRIVE LOCATIONS FOR SAVED FOLD 3
#
# NO TRAINING
# NO OUTER EVALUATION
# ============================================================

from pathlib import Path
import os

print("=" * 100)
print("SEARCHING FOR A02 FOLD 3 ARTIFACTS")
print("=" * 100)


SEARCH_ROOTS = [

    Path("/content/drive/MyDrive"),

    Path(
        "/content/drive/"
        ".shortcut-targets-by-id/"
        "1un2SaLv7_DvXerUlPtzKSNZ_j-mu1qFQ"
    ),
]


# Files that uniquely identify a useful completed fold
TARGET_FILENAMES = {

    "stage1_no_shrinkage.npz",

    "stage2_shrinkage.npz",

    "fold_training_complete.json",

    "best.weights.h5",
}


found = []


for root in SEARCH_ROOTS:

    if not root.exists():

        continue


    print("\nSearching:")
    print(root)


    for current_root, dirs, files in os.walk(root):

        path_text = current_root.lower()


        # Only inspect paths plausibly related to A02 / confirmatory CV
        if (
            "a02" not in path_text
            and
            "paper_confirmatory_cv_v1" not in path_text
        ):

            continue


        # We specifically care about Fold 3
        if (
            "outer_fold_3"
            not in path_text
        ):

            continue


        for filename in files:

            if filename in TARGET_FILENAMES:

                full_path = (
                    Path(current_root)
                    /
                    filename
                )

                found.append(
                    full_path
                )


print(
    "\n"
    + "=" * 100
)

print(
    "SEARCH RESULT"
)

print(
    "=" * 100
)


if not found:

    print(
        "\nNO A02 FOLD 3 ARTIFACTS FOUND."
    )

else:

    for path in sorted(
        found
    ):

        print(path)


# ------------------------------------------------------------
# Summarize candidate Fold-3 roots
# ------------------------------------------------------------

candidate_fold_roots = set()


for path in found:

    parts = list(
        path.parts
    )


    for i, part in enumerate(
        parts
    ):

        if part == "outer_fold_3":

            candidate_fold_roots.add(
                Path(
                    *parts[
                        :i+1
                    ]
                )
            )


print(
    "\nCandidate Fold-3 roots:"
)


if not candidate_fold_roots:

    print("NONE")

else:

    for root in sorted(
        candidate_fold_roots
    ):

        print("\n", root)


        stage1 = (
            root
            / "glass"
            / "stage1_no_shrinkage.npz"
        )


        stage2 = (
            root
            / "glass"
            / "stage2_shrinkage.npz"
        )


        checkpoint_count = len(
            list(
                root.rglob(
                    "best.weights.h5"
                )
            )
        )


        summary_count = len(
            list(
                root.rglob(
                    "training_summary.json"
                )
            )
        )


        print(
            "  GLASS Stage1:",
            stage1.is_file()
        )

        print(
            "  GLASS Stage2:",
            stage2.is_file()
        )

        print(
            "  checkpoints:",
            checkpoint_count,
            "/12"
        )

        print(
            "  summaries:",
            summary_count,
            "/12"
        )


print(
    "\nTraining performed: NO"
)

print(
    "Outer evaluation performed: NO"
)

In [ ]:
# ============================================================
# A02 — STEP 3
# FAST LOCAL TRAINING WORKSPACE
# + MODEL-LEVEL DRIVE PERSISTENCE
#
# NO TRAINING YET
# NO OUTER EVALUATION
# ============================================================

from pathlib import Path
import shutil
import json
import os


# ============================================================
# 1. SOURCE / PERSISTENT ROOTS
# ============================================================

DRIVE_PROJECT_ROOT = Path(
    "/content/drive/"
    ".shortcut-targets-by-id/"
    "1un2SaLv7_DvXerUlPtzKSNZ_j-mu1qFQ/"
    "EEG_GANet_Reproduction"
)

DRIVE_CV_ROOT = (
    DRIVE_PROJECT_ROOT
    / "GLASS_GANet"
    / "results"
    / "paper_confirmatory_cv_v1"
)

DRIVE_A02_ROOT = (
    DRIVE_CV_ROOT
    / "subject_runs"
    / "A02"
)

DRIVE_PREPARED = (
    DRIVE_PROJECT_ROOT
    / "GLASS_GANet"
    / "prepared_data"
    / "A02_GLASS_preprocessed_seed42.npz"
)


assert DRIVE_PROJECT_ROOT.is_dir()
assert DRIVE_CV_ROOT.is_dir()
assert DRIVE_A02_ROOT.is_dir()
assert DRIVE_PREPARED.is_file()


# ============================================================
# 2. FAST LOCAL ROOT
# ============================================================

LOCAL_ROOT = Path(
    "/content/A02_fast_run"
)

LOCAL_A02_ROOT = (
    LOCAL_ROOT
    / "subject_runs"
    / "A02"
)

LOCAL_PREPARED_DIR = (
    LOCAL_ROOT
    / "prepared_data"
)

LOCAL_PREPARED = (
    LOCAL_PREPARED_DIR
    / "A02_GLASS_preprocessed_seed42.npz"
)


LOCAL_A02_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

LOCAL_PREPARED_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 3. COPY PREPARED A02 TO LOCAL SSD
# ============================================================

if not LOCAL_PREPARED.is_file():

    print(
        "Copying prepared A02 data "
        "from Drive -> local SSD..."
    )

    shutil.copy2(
        DRIVE_PREPARED,
        LOCAL_PREPARED
    )


assert LOCAL_PREPARED.is_file()

assert (
    LOCAL_PREPARED.stat().st_size
    ==
    DRIVE_PREPARED.stat().st_size
)


print(
    "Prepared data local copy: PASS"
)


# ============================================================
# 4. RESTORE ANY EXISTING A02 FOLD CONTENT
#
# Folds 1/2 will be copied.
# Future partially completed Fold 3–7 will also be restored
# automatically after a runtime restart.
# ============================================================

def restore_fold_from_drive(
    fold
):

    src = (
        DRIVE_A02_ROOT
        / f"outer_fold_{fold}"
    )

    dst = (
        LOCAL_A02_ROOT
        / f"outer_fold_{fold}"
    )


    if not src.is_dir():

        print(
            f"Fold {fold}: "
            "nothing on Drive"
        )

        return


    if dst.exists():

        shutil.rmtree(
            dst
        )


    shutil.copytree(
        src,
        dst
    )


    print(
        f"Fold {fold}: "
        "restored Drive -> local"
    )


for fold in range(
    1,
    8
):

    restore_fold_from_drive(
        fold
    )


# ============================================================
# 5. ATOMIC COPY HELPERS
#
# These will be called after EACH completed training unit.
# ============================================================

def copy_file_atomic(
    source,
    destination
):

    source = Path(
        source
    )

    destination = Path(
        destination
    )

    assert source.is_file(), source


    destination.parent.mkdir(
        parents=True,
        exist_ok=True
    )


    temporary = (
        destination.parent
        /
        (
            destination.name
            +
            ".tmp"
        )
    )


    shutil.copy2(
        source,
        temporary
    )


    assert temporary.is_file()

    assert (
        temporary.stat().st_size
        ==
        source.stat().st_size
    )


    temporary.replace(
        destination
    )


    assert destination.is_file()


def persist_glass_file(
    fold,
    filename
):

    local_file = (
        LOCAL_A02_ROOT
        / f"outer_fold_{fold}"
        / "glass"
        / filename
    )

    drive_file = (
        DRIVE_A02_ROOT
        / f"outer_fold_{fold}"
        / "glass"
        / filename
    )


    copy_file_atomic(
        local_file,
        drive_file
    )


    print(
        f"Fold {fold} | "
        f"{filename}: "
        "PERSISTED TO DRIVE"
    )


def persist_model_run(
    fold,
    seed,
    model_name
):

    local_dir = (
        LOCAL_A02_ROOT
        / f"outer_fold_{fold}"
        / f"seed_{seed}"
        / model_name
    )

    drive_dir = (
        DRIVE_A02_ROOT
        / f"outer_fold_{fold}"
        / f"seed_{seed}"
        / model_name
    )


    weights = (
        local_dir
        / "best.weights.h5"
    )

    summary = (
        local_dir
        / "training_summary.json"
    )


    assert weights.is_file(), weights
    assert summary.is_file(), summary


    # Validate summary before persistence
    record = json.loads(
        summary.read_text(
            encoding="utf-8"
        )
    )


    assert (
        record.get("status")
        ==
        "completed"
    )

    assert (
        int(
            record["outer_fold"]
        )
        ==
        fold
    )

    assert (
        int(
            record["seed"]
        )
        ==
        seed
    )

    assert (
        record["model"]
        ==
        model_name
    )


    drive_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    # Save every file in this completed model directory
    for source_file in local_dir.iterdir():

        if source_file.is_file():

            copy_file_atomic(

                source_file,

                drive_dir
                /
                source_file.name
            )


    # Verify critical persistent files
    assert (
        drive_dir
        / "best.weights.h5"
    ).is_file()

    assert (
        drive_dir
        / "training_summary.json"
    ).is_file()


    print(
        f"Fold {fold} | "
        f"seed {seed} | "
        f"{model_name}: "
        "PERSISTED TO DRIVE"
    )


def persist_fold_marker(
    fold
):

    local_marker = (
        LOCAL_A02_ROOT
        / f"outer_fold_{fold}"
        / "fold_training_complete.json"
    )

    drive_marker = (
        DRIVE_A02_ROOT
        / f"outer_fold_{fold}"
        / "fold_training_complete.json"
    )


    copy_file_atomic(
        local_marker,
        drive_marker
    )


    print(
        f"Fold {fold}: "
        "COMPLETION MARKER PERSISTED"
    )


# ============================================================
# 6. MODEL-LEVEL RESUME AUDIT
# ============================================================

MODELS = [
    "structured_dbnet",
    "glass_gated_dbnet",
    "interaction_dbnet",
    "binary_dbnet",
]


print(
    "\n"
    + "=" * 100
)

print(
    "A02 LOCAL RESTART STATE"
)

print(
    "=" * 100
)


for fold in range(
    1,
    8
):

    fold_dir = (
        LOCAL_A02_ROOT
        / f"outer_fold_{fold}"
    )


    stage1 = (
        fold_dir
        / "glass"
        / "stage1_no_shrinkage.npz"
    )

    stage2 = (
        fold_dir
        / "glass"
        / "stage2_shrinkage.npz"
    )


    completed = []


    for seed in [
        1,
        2,
        3
    ]:

        for model_name in MODELS:

            run_dir = (
                fold_dir
                / f"seed_{seed}"
                / model_name
            )


            w = (
                run_dir
                / "best.weights.h5"
            )

            s = (
                run_dir
                / "training_summary.json"
            )


            if (
                w.is_file()
                and
                s.is_file()
            ):

                try:

                    r = json.loads(
                        s.read_text(
                            encoding="utf-8"
                        )
                    )

                    if (
                        r.get("status")
                        ==
                        "completed"
                    ):

                        completed.append(
                            (
                                seed,
                                model_name
                            )
                        )

                except Exception:

                    pass


    print(
        f"\nFold {fold}:"
    )

    print(
        "  GLASS Stage1:",
        "PASS"
        if stage1.is_file()
        else "NO"
    )

    print(
        "  GLASS Stage2:",
        "PASS"
        if stage2.is_file()
        else "NO"
    )

    print(
        f"  reusable models: "
        f"{len(completed)}/12"
    )


# ============================================================
# 7. FINAL
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "A02 FAST LOCAL + DRIVE PERSISTENCE SETUP: PASS"
)

print(
    "=" * 100
)

print(
    "\nLOCAL_A02_ROOT:"
)

print(
    LOCAL_A02_ROOT
)

print(
    "\nDRIVE_A02_ROOT:"
)

print(
    DRIVE_A02_ROOT
)

print(
    "\nTraining performed: NO"
)

print(
    "Outer evaluation performed: NO"
)

print(
    "\nCompleted models will be saved "
    "to Drive individually."
)

In [ ]:
# ============================================================
# A02 — STEP 4
# LOCAL SOURCE + FROZEN MODEL DEFINITIONS + PARAMETER AUDIT
#
# NO TRAINING
# NO OUTER EVALUATION
# ============================================================

from pathlib import Path
import shutil
import hashlib
import sys
import importlib
import gc

import numpy as np
import tensorflow as tf


# ============================================================
# 1. REQUIRED PREVIOUS STATE
# ============================================================

for name in [
    "DRIVE_PROJECT_ROOT",
    "LOCAL_ROOT",
    "LOCAL_A02_ROOT",
    "LOCAL_PREPARED",
]:
    assert name in globals(), (
        f"Missing {name}. "
        "Run Steps 1–3 first."
    )


assert LOCAL_PREPARED.is_file()


# ============================================================
# 2. COPY EEG-GANET SOURCE TO LOCAL SSD
# ============================================================

DRIVE_CODE_DIR = (
    DRIVE_PROJECT_ROOT
    / "code"
    / "EEG-GANet"
)

LOCAL_CODE_DIR = (
    LOCAL_ROOT
    / "code"
    / "EEG-GANet"
)


required_source = [
    "github_model.py",
    "github_Utils.py",
    "GANs.py",
]


for filename in required_source:

    source = (
        DRIVE_CODE_DIR
        / filename
    )

    assert source.is_file(), (
        f"Missing source:\n{source}"
    )


if LOCAL_CODE_DIR.exists():

    shutil.rmtree(
        LOCAL_CODE_DIR
    )


shutil.copytree(
    DRIVE_CODE_DIR,
    LOCAL_CODE_DIR
)


print(
    "EEG-GANet source copied "
    "Drive -> local SSD: PASS"
)


# ============================================================
# 3. COPY GLASS TF2.20 COMPATIBILITY SOURCE LOCALLY
# ============================================================

DRIVE_GLASS_TF220 = (
    DRIVE_PROJECT_ROOT
    / "GLASS_GANet"
    / "integration"
    / "glass_tf220.py"
)

assert DRIVE_GLASS_TF220.is_file(), (
    DRIVE_GLASS_TF220
)


LOCAL_INTEGRATION_DIR = (
    LOCAL_ROOT
    / "integration"
)

LOCAL_INTEGRATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)


LOCAL_GLASS_TF220 = (
    LOCAL_INTEGRATION_DIR
    / "glass_tf220.py"
)


shutil.copy2(
    DRIVE_GLASS_TF220,
    LOCAL_GLASS_TF220
)


assert LOCAL_GLASS_TF220.is_file()


print(
    "GLASS TF2.20 source copied "
    "Drive -> local SSD: PASS"
)


# ============================================================
# 4. SOURCE HASH
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        for block in iter(
            lambda:
                f.read(
                    1024 * 1024
                ),
            b"",
        ):

            h.update(
                block
            )

    return h.hexdigest()


EXPECTED_DBNET_HASH = (
    "f3e570d12de34080"
    "ea13003a37bf9a24"
    "93b93f1a2376631e"
    "8224a40942231cd5"
)


LOCAL_DBNET = (
    LOCAL_CODE_DIR
    / "github_model.py"
)


actual_hash = sha256_file(
    LOCAL_DBNET
)


print(
    "\nDBNet SHA256:"
)

print(
    actual_hash
)


assert (
    actual_hash
    ==
    EXPECTED_DBNET_HASH
), (
    "Frozen DBNet source hash mismatch."
)


print(
    "Frozen DBNet source hash: PASS"
)


# ============================================================
# 5. IMPORT ORIGINAL EEG-GANET SOURCE
# ============================================================

if str(
    LOCAL_CODE_DIR
) not in sys.path:

    sys.path.insert(
        0,
        str(
            LOCAL_CODE_DIR
        )
    )


for module_name in [
    "GANs",
    "github_Utils",
    "github_model",
]:

    sys.modules.pop(
        module_name,
        None
    )


GANs = importlib.import_module(
    "GANs"
)

github_Utils = importlib.import_module(
    "github_Utils"
)

github_model = importlib.import_module(
    "github_model"
)


EEG_DBNet_V2 = (
    github_model
    .EEG_DBNet_V2
)


print(
    "\nEEG-GANet import chain: PASS"
)


# ============================================================
# 6. IMPORT LOCAL GLASS
# ============================================================

import importlib.util


spec = (
    importlib.util
    .spec_from_file_location(
        "glass_tf220_A02",
        LOCAL_GLASS_TF220
    )
)


glass_module = (
    importlib.util
    .module_from_spec(
        spec
    )
)


spec.loader.exec_module(
    glass_module
)


Glass = (
    glass_module.Glass
)


print(
    "GLASS TF2.20 import: PASS"
)


# ============================================================
# 7. FROZEN CONSTANTS
# ============================================================

LEARNING_RATE = 0.0009

MAX_EPOCHS = 1000

PATIENCE = 50

MIN_DELTA = 1e-4

GROUP_BATCH_SIZE = 32

BINARY_BATCH_SIZE = 64


EXPECTED_PARAMS = {

    "structured_dbnet":
        3961,

    "glass_gated_dbnet":
        4986,

    "interaction_dbnet":
        3989,

    "binary_dbnet":
        4090,
}


# ============================================================
# 8. BINARY TARGET AUC
# ============================================================

class TargetAUC(
    tf.keras.metrics.AUC
):

    def __init__(
        self,
        name="target_auc",
        **kwargs
    ):

        super().__init__(
            name=name,
            curve="ROC",
            **kwargs
        )


    def update_state(
        self,
        y_true,
        y_pred,
        sample_weight=None
    ):

        return super().update_state(
            y_true[:, 1],
            y_pred[:, 1],
            sample_weight=
                sample_weight
        )


# ============================================================
# 9. SHARED DBNET EPOCH SCORER
# ============================================================

def build_epoch_scorer():

    wrapper = EEG_DBNet_V2(
        NumFilter=8,
        SamplingFrequency=128,
        NumChannels=8,
        FilterScaler=2,
        NumClasses=2,
        DropoutRate=0.5,
    )


    public_model = (
        wrapper.build_model()
    )


    assert isinstance(
        public_model.layers[-3],
        tf.keras.layers.Concatenate
    )


    feature_extractor = (
        tf.keras.Model(
            inputs=
                public_model.input,

            outputs=
                public_model
                .layers[-3]
                .output,

            name=
                "dbnet_v2_feature_extractor",
        )
    )


    x = tf.keras.Input(
        shape=(
            8,
            128,
            1
        )
    )


    features = (
        feature_extractor(
            x
        )
    )


    score = (
        tf.keras.layers.Dense(

            1,

            kernel_constraint=
                tf.keras.constraints
                .max_norm(
                    0.25
                ),

            name=
                "stimulus_score",

        )(
            features
        )
    )


    return tf.keras.Model(
        x,
        score,
        name=
            "dbnet_v2_epoch_scorer"
    )


# ============================================================
# 10. STRUCTURED DBNET
# ============================================================

def build_structured_dbnet():

    scorer = (
        build_epoch_scorer()
    )


    group_input = (
        tf.keras.Input(
            shape=(
                6,
                8,
                128,
                1
            )
        )
    )


    scores = (
        tf.keras.layers
        .TimeDistributed(
            scorer,
            name=
                "shared_dbnet_scorer"
        )(
            group_input
        )
    )


    logits = (
        tf.keras.layers.Reshape(
            (6,),
            name=
                "dbnet_logits"
        )(
            scores
        )
    )


    output = (
        tf.keras.layers.Softmax(
            axis=-1,
            name=
                "six_choice_probabilities"
        )(
            logits
        )
    )


    model = tf.keras.Model(
        group_input,
        output,
        name=
            "structured_dbnet_v2"
    )


    model.compile(

        optimizer=
            tf.keras.optimizers.Adam(
                learning_rate=
                    LEARNING_RATE
            ),

        loss=
            "categorical_crossentropy",

        metrics=[
            tf.keras.metrics
            .CategoricalAccuracy(
                name=
                    "six_choice_accuracy"
            )
        ],
    )


    return model


# ============================================================
# 11. FIXED GLASS SCORE LAYER
# ============================================================

class FixedGlassScores(
    tf.keras.layers.Layer
):

    def __init__(
        self,
        beta_matrix,
        **kwargs
    ):

        super().__init__(
            trainable=False,
            **kwargs
        )


        beta_matrix = np.asarray(
            beta_matrix,
            dtype=np.float32
        )


        assert beta_matrix.shape == (
            8,
            128
        )


        self.initial_beta = (
            beta_matrix
        )


    def build(
        self,
        input_shape
    ):

        self.beta_matrix = (
            self.add_weight(

                name=
                    "beta_matrix",

                shape=(
                    8,
                    128
                ),

                initializer=
                    tf.keras.initializers
                    .Constant(
                        self.initial_beta
                    ),

                trainable=False,
            )
        )


    def call(
        self,
        inputs
    ):

        return tf.einsum(
            "bkct,ct->bk",

            inputs[..., 0],

            self.beta_matrix
        )


# ============================================================
# 12. NONNEGATIVE GLASS GATE
# ============================================================

class NonnegativeGlassResidual(
    tf.keras.layers.Layer
):

    def build(
        self,
        input_shape
    ):

        self.glass_gate = (
            self.add_weight(

                name=
                    "glass_gate",

                shape=(),

                initializer=
                    "zeros",

                trainable=True,

                constraint=
                    tf.keras.constraints
                    .NonNeg(),
            )
        )


    def call(
        self,
        inputs
    ):

        dbnet_logits, glass_logits = (
            inputs
        )


        centered = (
            glass_logits
            -
            tf.reduce_mean(
                glass_logits,
                axis=-1,
                keepdims=True
            )
        )


        scale = (
            tf.math.reduce_std(
                centered,
                axis=-1,
                keepdims=True
            )
        )


        normalized = (
            centered
            /
            (
                scale
                +
                1e-6
            )
        )


        return (
            dbnet_logits
            +
            self.glass_gate
            *
            normalized
        )


# ============================================================
# 13. GLASS-GATED STRUCTURED DBNET
# ============================================================

def build_gated_hybrid(
    beta_matrix
):

    scorer = (
        build_epoch_scorer()
    )


    x = tf.keras.Input(
        shape=(
            6,
            8,
            128,
            1
        )
    )


    dbnet_scores = (
        tf.keras.layers
        .TimeDistributed(
            scorer,
            name=
                "shared_dbnet_scorer"
        )(
            x
        )
    )


    dbnet_logits = (
        tf.keras.layers.Reshape(
            (6,),
            name=
                "dbnet_logits"
        )(
            dbnet_scores
        )
    )


    glass_logits = (
        FixedGlassScores(
            beta_matrix,
            name=
                "fixed_official_glass_scores"
        )(
            x
        )
    )


    combined = (
        NonnegativeGlassResidual(
            name=
                "nonnegative_glass_residual"
        )(
            [
                dbnet_logits,
                glass_logits
            ]
        )
    )


    output = (
        tf.keras.layers.Softmax(
            axis=-1,
            name=
                "six_choice_probabilities"
        )(
            combined
        )
    )


    model = tf.keras.Model(
        x,
        output,
        name=
            "glass_gated_structured_dbnet"
    )


    model.compile(

        optimizer=
            tf.keras.optimizers.Adam(
                learning_rate=
                    LEARNING_RATE
            ),

        loss=
            "categorical_crossentropy",

        metrics=[
            tf.keras.metrics
            .CategoricalAccuracy(
                name=
                    "six_choice_accuracy"
            )
        ],
    )


    return model


# ============================================================
# 14. FISHER-z FEATURES
# ============================================================

CHANNEL_PAIRS = np.asarray(
    [
        (u, v)
        for u in range(7)
        for v in range(
            u + 1,
            8
        )
    ],
    dtype=np.int64
)


assert CHANNEL_PAIRS.shape == (
    28,
    2
)


def compute_fisher_features(
    grouped_eeg
):

    eeg = np.asarray(
        grouped_eeg,
        dtype=np.float64
    )[..., 0]


    groups = (
        eeg.shape[0]
    )


    flat = eeg.reshape(
        groups * 6,
        8,
        128
    )


    if not np.all(
        np.std(
            flat,
            axis=-1
        ) > 0
    ):

        raise RuntimeError(
            "Zero variance epoch."
        )


    features = np.empty(
        (
            flat.shape[0],
            28
        ),
        dtype=np.float64
    )


    for idx, (
        a,
        b
    ) in enumerate(
        CHANNEL_PAIRS
    ):

        xa = flat[:, a, :]
        xb = flat[:, b, :]


        xa = (
            xa
            -
            xa.mean(
                axis=1,
                keepdims=True
            )
        )


        xb = (
            xb
            -
            xb.mean(
                axis=1,
                keepdims=True
            )
        )


        r = (

            (
                xa
                *
                xb
            ).sum(
                axis=1
            )

            /

            np.sqrt(

                (
                    xa ** 2
                ).sum(
                    axis=1
                )

                *

                (
                    xb ** 2
                ).sum(
                    axis=1
                )
            )
        )


        features[
            :,
            idx
        ] = np.arctanh(
            np.clip(
                r,
                -0.999999,
                0.999999
            )
        )


    return (
        features.reshape(
            groups,
            6,
            28
        )
        .astype(
            np.float32
        )
    )


# ============================================================
# 15. INTERACTION DBNET
# ============================================================

def build_interaction_dbnet():

    scorer = (
        build_epoch_scorer()
    )


    eeg_input = (
        tf.keras.Input(
            shape=(
                6,
                8,
                128,
                1
            )
        )
    )


    interaction_input = (
        tf.keras.Input(
            shape=(
                6,
                28
            )
        )
    )


    dbnet_scores = (
        tf.keras.layers
        .TimeDistributed(
            scorer,
            name=
                "shared_dbnet_scorer"
        )(
            eeg_input
        )
    )


    dbnet_logits = (
        tf.keras.layers.Reshape(
            (6,)
        )(
            dbnet_scores
        )
    )


    interaction_scores = (
        tf.keras.layers
        .TimeDistributed(

            tf.keras.layers.Dense(
                1,

                use_bias=False,

                kernel_initializer=
                    "zeros",

                name=
                    "interaction_linear"
            ),

            name=
                "shared_rtgp_interaction_scorer"

        )(
            interaction_input
        )
    )


    interaction_logits = (
        tf.keras.layers.Reshape(
            (6,)
        )(
            interaction_scores
        )
    )


    interaction_logits = (
        tf.keras.layers.Rescaling(
            scale=
                1.0 / 28.0
        )(
            interaction_logits
        )
    )


    combined = (
        tf.keras.layers.Add()(
            [
                dbnet_logits,
                interaction_logits
            ]
        )
    )


    output = (
        tf.keras.layers.Softmax(
            axis=-1
        )(
            combined
        )
    )


    model = tf.keras.Model(
        [
            eeg_input,
            interaction_input
        ],
        output
    )


    model.compile(

        optimizer=
            tf.keras.optimizers.Adam(
                learning_rate=
                    LEARNING_RATE
            ),

        loss=
            "categorical_crossentropy",

        metrics=[
            tf.keras.metrics
            .CategoricalAccuracy(
                name=
                    "six_choice_accuracy"
            )
        ],
    )


    return model


# ============================================================
# 16. BINARY DBNET
# ============================================================

def build_binary_dbnet_frozen():

    wrapper = EEG_DBNet_V2(
        NumFilter=8,
        SamplingFrequency=128,
        NumChannels=8,
        FilterScaler=2,
        NumClasses=2,
        DropoutRate=0.5,
    )


    model = (
        wrapper.build_model()
    )


    model.compile(

        optimizer=
            tf.keras.optimizers.Adam(
                learning_rate=
                    LEARNING_RATE
            ),

        loss=
            "categorical_crossentropy",

        metrics=[
            TargetAUC(
                name=
                    "target_auc"
            )
        ],
    )


    return model


# ============================================================
# 17. HARD PARAMETER AUDIT
# ============================================================

dummy_beta = np.zeros(
    (
        8,
        128
    ),
    dtype=np.float32
)


builders = {

    "structured_dbnet":
        build_structured_dbnet,

    "glass_gated_dbnet":
        lambda:
            build_gated_hybrid(
                dummy_beta
            ),

    "interaction_dbnet":
        build_interaction_dbnet,

    "binary_dbnet":
        build_binary_dbnet_frozen,
}


print(
    "\n"
    + "=" * 100
)

print(
    "A02 FROZEN MODEL PARAMETER AUDIT"
)

print(
    "=" * 100
)


for name, builder in (
    builders.items()
):

    tf.keras.backend.clear_session()

    gc.collect()

    tf.keras.utils.set_random_seed(
        1
    )


    model = builder()


    total = int(
        model.count_params()
    )


    trainable = int(
        sum(
            np.prod(
                w.shape
            )
            for w
            in model.trainable_weights
        )
    )


    nontrainable = int(
        sum(
            np.prod(
                w.shape
            )
            for w
            in model.non_trainable_weights
        )
    )


    print(
        f"{name:24s} | "
        f"total={total:4d} | "
        f"trainable={trainable:4d} | "
        f"nontrainable={nontrainable:4d}"
    )


    assert (
        total
        ==
        EXPECTED_PARAMS[
            name
        ]
    )


    del model


tf.keras.backend.clear_session()

gc.collect()


# ============================================================
# 18. FINAL STATUS
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "A02 LOCAL SOURCE + FROZEN MODEL SETUP: PASS"
)

print(
    "=" * 100
)

print(
    "Structured:   3961 PASS"
)

print(
    "Gated:        4986 PASS"
)

print(
    "Interaction:  3989 PASS"
)

print(
    "Binary:       4090 PASS"
)

print(
    "\nTraining performed: NO"
)

print(
    "Outer evaluation performed: NO"
)

In [ ]:
# ============================================================
# A02 — STEP 5
# FAST RESTART-SAFE TRAINING
#
# TRAINS ONLY FOLDS 3–7
#
# Local SSD:
#   /content/A02_fast_run
#
# Persistent Drive:
#   .../paper_confirmatory_cv_v1/subject_runs/A02
#
# Persistence granularity:
#   GLASS Stage 1
#   GLASS Stage 2
#   every individual neural model
#   final fold marker
#
# OUTER EEG/LABELS ARE NOT ACCESSED.
# OUTER EVALUATION IS NOT PERFORMED.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import shutil
import time
import gc

import numpy as np
import pandas as pd
import tensorflow as tf


# ============================================================
# 0. REQUIRED STATE
# ============================================================

REQUIRED = [
    "LOCAL_A02_ROOT",
    "DRIVE_A02_ROOT",
    "LOCAL_PREPARED",
    "Glass",
    "build_structured_dbnet",
    "build_gated_hybrid",
    "build_interaction_dbnet",
    "build_binary_dbnet_frozen",
    "compute_fisher_features",
    "EXPECTED_PARAMS",
]

for name in REQUIRED:

    assert name in globals(), (
        f"Missing {name}. "
        "Run Steps 1–4 first."
    )


SUBJECT = "A02"

SEEDS = [
    1,
    2,
    3
]

FOLDS_TO_RUN = [
    3,
    4,
    5,
    6,
    7
]


# ============================================================
# 1. EXACT FROZEN FOLDS
# ============================================================

FOLDS = {

    1: {
        "train": [
            1,3,4,6,7,9,10,11,13,14,
            15,16,17,19,20,21,24,25,27,28,
            29,30,31,32,34
        ],
        "val": [
            2,5,12,33,35
        ],
        "outer": [
            8,18,22,23,26
        ],
    },

    2: {
        "train": [
            1,3,4,6,7,8,9,11,13,14,
            15,16,17,18,20,21,22,23,24,26,
            28,29,30,31,34
        ],
        "val": [
            10,19,25,27,32
        ],
        "outer": [
            2,5,12,33,35
        ],
    },

    3: {
        "train": [
            1,2,3,5,6,8,11,12,13,14,
            15,16,18,20,21,22,23,24,26,28,
            29,30,33,34,35
        ],
        "val": [
            4,7,9,17,31
        ],
        "outer": [
            10,19,25,27,32
        ],
    },

    4: {
        "train": [
            1,2,3,5,8,10,11,12,13,14,
            18,19,20,21,22,23,25,26,27,28,
            30,32,33,34,35
        ],
        "val": [
            6,15,16,24,29
        ],
        "outer": [
            4,7,9,17,31
        ],
    },

    5: {
        "train": [
            1,2,3,4,5,7,8,9,10,12,
            13,14,17,18,19,22,23,25,26,27,
            30,31,32,33,35
        ],
        "val": [
            11,20,21,28,34
        ],
        "outer": [
            6,15,16,24,29
        ],
    },

    6: {
        "train": [
            2,4,5,6,7,8,9,10,12,15,
            16,17,18,19,22,23,24,25,26,27,
            29,31,32,33,35
        ],
        "val": [
            1,3,13,14,30
        ],
        "outer": [
            11,20,21,28,34
        ],
    },

    7: {
        "train": [
            2,4,5,6,7,9,10,11,12,15,
            16,17,19,20,21,24,25,27,28,29,
            31,32,33,34,35
        ],
        "val": [
            8,18,22,23,26
        ],
        "outer": [
            1,3,13,14,30
        ],
    },
}


# Hard split audit

outer_all = []

for fold, split in FOLDS.items():

    train = set(split["train"])
    val = set(split["val"])
    outer = set(split["outer"])

    assert len(train) == 25
    assert len(val) == 5
    assert len(outer) == 5

    assert not (train & val)
    assert not (train & outer)
    assert not (val & outer)

    assert (
        train | val | outer
    ) == set(range(1, 36))

    outer_all.extend(
        split["outer"]
    )


assert sorted(
    outer_all
) == list(range(1, 36))

print("Frozen fold audit: PASS")


# ============================================================
# 2. FROZEN TRAINING CONSTANTS
# ============================================================

LEARNING_RATE = 0.0009

MAX_EPOCHS = 1000

PATIENCE = 50

MIN_DELTA = 1e-4

GROUP_BATCH_SIZE = 32

BINARY_BATCH_SIZE = 64


GLASS_NUM_STEPS = 2000

GLASS_SAMPLE_SIZE = 10

GLASS_IMPORTANCE_SAMPLE_SIZE = 10

GLASS_POSTERIOR_SAMPLE_SIZE = 5000   # metadata only

GLASS_LEARNING_RATE = 0.05

GLASS_SEED = 1


STRUCTURED_ORDER = {

    1: [
        "structured_dbnet",
        "glass_gated_dbnet",
        "interaction_dbnet",
    ],

    2: [
        "interaction_dbnet",
        "glass_gated_dbnet",
        "structured_dbnet",
    ],

    3: [
        "structured_dbnet",
        "glass_gated_dbnet",
        "interaction_dbnet",
    ],
}


# ============================================================
# 3. SHA-256 + STRICT DRIVE COPY
# ============================================================

def sha256_file(path):

    path = Path(path)

    h = hashlib.sha256()

    with path.open("rb") as f:

        for block in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):

            h.update(block)

    return h.hexdigest()


def copy_file_verified(
    source,
    destination,
):

    source = Path(source)

    destination = Path(
        destination
    )

    assert source.is_file(), source

    destination.parent.mkdir(
        parents=True,
        exist_ok=True
    )


    temp = (
        destination.parent
        /
        (
            destination.name
            +
            ".uploading"
        )
    )


    if temp.exists():
        temp.unlink()


    source_sha = sha256_file(
        source
    )


    shutil.copy2(
        source,
        temp
    )


    assert temp.is_file()


    temp_sha = sha256_file(
        temp
    )


    assert (
        temp_sha
        ==
        source_sha
    ), (
        f"Temporary Drive copy SHA mismatch:\n"
        f"{source}"
    )


    temp.replace(
        destination
    )


    assert destination.is_file()


    destination_sha = (
        sha256_file(
            destination
        )
    )


    assert (
        destination_sha
        ==
        source_sha
    ), (
        f"Final Drive copy SHA mismatch:\n"
        f"{destination}"
    )


    return source_sha


# ============================================================
# 4. ATOMIC LOCAL JSON
# ============================================================

def atomic_json_save(
    path,
    payload,
):

    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )


    temp = (
        path.parent
        /
        (
            path.name
            +
            ".tmp"
        )
    )


    temp.write_text(
        json.dumps(
            payload,
            indent=2,
            allow_nan=False,
        ),
        encoding="utf-8"
    )


    temp.replace(
        path
    )


# ============================================================
# 5. STRICT PERSISTENCE HELPERS
# ============================================================

def persist_glass_file_strict(
    fold,
    filename,
):

    local_file = (
        LOCAL_A02_ROOT
        / f"outer_fold_{fold}"
        / "glass"
        / filename
    )


    drive_file = (
        DRIVE_A02_ROOT
        / f"outer_fold_{fold}"
        / "glass"
        / filename
    )


    digest = copy_file_verified(
        local_file,
        drive_file
    )


    print(
        f"    DRIVE VERIFIED | "
        f"Fold {fold} | "
        f"{filename}"
    )

    print(
        f"    SHA256: "
        f"{digest[:16]}..."
    )


def persist_model_strict(
    fold,
    seed,
    model_name,
):

    local_dir = (
        LOCAL_A02_ROOT
        / f"outer_fold_{fold}"
        / f"seed_{seed}"
        / model_name
    )


    drive_dir = (
        DRIVE_A02_ROOT
        / f"outer_fold_{fold}"
        / f"seed_{seed}"
        / model_name
    )


    weights = (
        local_dir
        / "best.weights.h5"
    )

    summary = (
        local_dir
        / "training_summary.json"
    )


    assert weights.is_file()
    assert summary.is_file()


    record = json.loads(
        summary.read_text(
            encoding="utf-8"
        )
    )


    assert (
        record["status"]
        ==
        "completed"
    )

    assert (
        record["subject"]
        ==
        SUBJECT
    )

    assert (
        int(record["outer_fold"])
        ==
        fold
    )

    assert (
        int(record["seed"])
        ==
        seed
    )

    assert (
        record["model"]
        ==
        model_name
    )


    drive_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    for source_file in sorted(
        local_dir.iterdir()
    ):

        if source_file.is_file():

            copy_file_verified(
                source_file,
                drive_dir
                /
                source_file.name
            )


    # Final critical verification

    assert (
        sha256_file(
            weights
        )
        ==
        sha256_file(
            drive_dir
            /
            "best.weights.h5"
        )
    )


    assert (
        sha256_file(
            summary
        )
        ==
        sha256_file(
            drive_dir
            /
            "training_summary.json"
        )
    )


    print(
        f"    DRIVE VERIFIED | "
        f"Fold {fold} | "
        f"seed {seed} | "
        f"{model_name}"
    )


def persist_fold_marker_strict(
    fold,
):

    local = (
        LOCAL_A02_ROOT
        / f"outer_fold_{fold}"
        / "fold_training_complete.json"
    )


    drive = (
        DRIVE_A02_ROOT
        / f"outer_fold_{fold}"
        / "fold_training_complete.json"
    )


    copy_file_verified(
        local,
        drive
    )


    print(
        f"Fold {fold}: "
        "COMPLETION MARKER "
        "DRIVE VERIFIED"
    )


# ============================================================
# 6. LOAD PREPARED A02
# ============================================================

with np.load(
    LOCAL_PREPARED,
    allow_pickle=False
) as archive:

    X_ALL = np.asarray(
        archive[
            "X_glass_processed_grouped"
        ],
        dtype=np.float32
    )


    Y_ALL = np.asarray(
        archive[
            "y_grouped"
        ],
        dtype=np.float32
    )


assert X_ALL.shape == (
    35,
    10,
    2,
    6,
    8,
    128
)


assert Y_ALL.shape == (
    35,
    10,
    2,
    6
)


assert np.isfinite(
    X_ALL
).all()


print(
    "\nPrepared A02:",
    X_ALL.shape,
    Y_ALL.shape
)


# ============================================================
# 7. FOLD DATA — TRAIN + INNER VALIDATION ONLY
# ============================================================

def prepare_fold_data(
    fold,
):

    split = FOLDS[
        fold
    ]


    train_chars = list(
        split["train"]
    )

    val_chars = list(
        split["val"]
    )

    outer_chars = list(
        split["outer"]
    )


    train_idx = (
        np.asarray(
            train_chars,
            dtype=np.int64
        )
        -
        1
    )


    val_idx = (
        np.asarray(
            val_chars,
            dtype=np.int64
        )
        -
        1
    )


    # --------------------------------------------------------
    # IMPORTANT:
    # no outer index is used here.
    # --------------------------------------------------------

    X_train = (
        X_ALL[
            train_idx
        ]
        .reshape(
            500,
            6,
            8,
            128
        )[
            ...,
            None
        ]
        .astype(
            np.float32
        )
    )


    y_train = (
        Y_ALL[
            train_idx
        ]
        .reshape(
            500,
            6
        )
        .astype(
            np.float32
        )
    )


    X_val = (
        X_ALL[
            val_idx
        ]
        .reshape(
            100,
            6,
            8,
            128
        )[
            ...,
            None
        ]
        .astype(
            np.float32
        )
    )


    y_val = (
        Y_ALL[
            val_idx
        ]
        .reshape(
            100,
            6
        )
        .astype(
            np.float32
        )
    )


    assert X_train.shape == (
        500,
        6,
        8,
        128,
        1
    )


    assert X_val.shape == (
        100,
        6,
        8,
        128,
        1
    )


    assert np.all(
        y_train.sum(
            axis=1
        )
        ==
        1
    )


    assert np.all(
        y_val.sum(
            axis=1
        )
        ==
        1
    )


    return {

        "train_chars":
            train_chars,

        "val_chars":
            val_chars,

        "outer_chars":
            outer_chars,

        "X_train":
            X_train,

        "y_train":
            y_train,

        "X_val":
            X_val,

        "y_val":
            y_val,
    }


# ============================================================
# 8. MODEL COMPLETION CHECK
# ============================================================

def model_complete(
    fold,
    seed,
    model_name,
):

    run_dir = (
        LOCAL_A02_ROOT
        / f"outer_fold_{fold}"
        / f"seed_{seed}"
        / model_name
    )


    weights = (
        run_dir
        / "best.weights.h5"
    )


    summary = (
        run_dir
        / "training_summary.json"
    )


    if not (
        weights.is_file()
        and
        summary.is_file()
    ):

        return False


    try:

        record = json.loads(
            summary.read_text(
                encoding="utf-8"
            )
        )


        return (

            record.get("status")
            ==
            "completed"

            and

            record.get("subject")
            ==
            SUBJECT

            and

            int(
                record.get(
                    "outer_fold",
                    -1
                )
            )
            ==
            fold

            and

            int(
                record.get(
                    "seed",
                    -1
                )
            )
            ==
            seed

            and

            record.get("model")
            ==
            model_name

            and

            int(
                record.get(
                    "parameter_count",
                    -1
                )
            )
            ==
            EXPECTED_PARAMS[
                model_name
            ]
        )


    except Exception:

        return False


# ============================================================
# 9. GLASS — RESTART SAFE
# ============================================================

def run_glass(
    fold,
    fold_data,
):

    glass_dir = (
        LOCAL_A02_ROOT
        / f"outer_fold_{fold}"
        / "glass"
    )


    glass_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    stage1_path = (
        glass_dir
        / "stage1_no_shrinkage.npz"
    )


    stage2_path = (
        glass_dir
        / "stage2_shrinkage.npz"
    )


    X_train = np.asarray(
        fold_data[
            "X_train"
        ][..., 0],
        dtype=np.float32
    )


    y_train = np.asarray(
        fold_data[
            "y_train"
        ],
        dtype=np.float32
    )


    train_chars = np.asarray(
        fold_data[
            "train_chars"
        ],
        dtype=np.int64
    )


    # --------------------------------------------------------
    # Reuse Stage 2
    # --------------------------------------------------------

    if stage2_path.is_file():

        with np.load(
            stage2_path,
            allow_pickle=False
        ) as archive:

            beta = np.asarray(
                archive[
                    "betaMat"
                ],
                dtype=np.float32
            )

            saved_train = np.asarray(
                archive[
                    "training_characters_1based"
                ],
                dtype=np.int64
            )


        np.testing.assert_array_equal(
            saved_train,
            train_chars
        )


        assert beta.shape == (
            8,
            128
        )


        assert np.isfinite(
            beta
        ).all()


        print(
            "  GLASS Stage 2: REUSED"
        )


        # Ensure persistent copy exists/verified again

        persist_glass_file_strict(
            fold,
            "stage2_shrinkage.npz"
        )


        return beta


    # --------------------------------------------------------
    # Stage 1
    # --------------------------------------------------------

    if stage1_path.is_file():

        with np.load(
            stage1_path,
            allow_pickle=False
        ) as archive:

            beta_stage1 = np.asarray(
                archive[
                    "betaMat"
                ],
                dtype=np.float32
            )

            losses_stage1 = np.asarray(
                archive[
                    "losses"
                ],
                dtype=np.float32
            )

            stage1_seconds = float(
                archive[
                    "stage1_fitting_seconds"
                ]
            )

            saved_train = np.asarray(
                archive[
                    "training_characters_1based"
                ],
                dtype=np.int64
            )


        np.testing.assert_array_equal(
            saved_train,
            train_chars
        )


        print(
            "  GLASS Stage 1: REUSED"
        )


    else:

        print(
            "  GLASS Stage 1: TRAINING..."
        )


        tf.keras.utils.set_random_seed(
            GLASS_SEED
        )


        glass1 = Glass(
            shrinkage_factor=
                0.0,

            dtype=
                tf.float32
        )


        glass1.process_data(
            X_train,
            y_train
        )


        t0 = time.perf_counter()


        with tf.device(
            "/CPU:0"
        ):

            # IMPORTANT:
            # posterior_sample_size is metadata only.
            # It is NOT passed to mfvb().

            glass1.mfvb(

                num_steps=
                    GLASS_NUM_STEPS,

                sample_size=
                    GLASS_SAMPLE_SIZE,

                importance_sample_size=
                    GLASS_IMPORTANCE_SAMPLE_SIZE,

                learning_rate=
                    GLASS_LEARNING_RATE,

                seed=
                    GLASS_SEED,
            )


        stage1_seconds = (
            time.perf_counter()
            -
            t0
        )


        beta_stage1 = np.asarray(
            glass1.betaMat,
            dtype=np.float32
        )


        losses_stage1 = np.asarray(
            glass1.losses,
            dtype=np.float32
        )


        assert beta_stage1.shape == (
            8,
            128
        )


        cutoff = float(
            np.median(
                np.abs(
                    beta_stage1
                )
            )
        )


        temp = (
            glass_dir
            / "stage1_no_shrinkage.tmp"
        )


        with temp.open(
            "wb"
        ) as f:

            np.savez_compressed(

                f,

                betaMat=
                    beta_stage1,

                losses=
                    losses_stage1,

                cutoff=
                    np.asarray(
                        cutoff
                    ),

                shrinkage_factor=
                    np.asarray(
                        0.0
                    ),

                stage1_fitting_seconds=
                    np.asarray(
                        stage1_seconds
                    ),

                training_characters_1based=
                    train_chars,

                training_data_only_for_fit=
                    np.asarray(
                        True
                    ),

                inner_validation_used_for_fitting=
                    np.asarray(
                        False
                    ),

                outer_test_used_for_fitting=
                    np.asarray(
                        False
                    ),

                num_steps_per_stage=
                    np.asarray(
                        GLASS_NUM_STEPS
                    ),

                sample_size=
                    np.asarray(
                        GLASS_SAMPLE_SIZE
                    ),

                importance_sample_size=
                    np.asarray(
                        GLASS_IMPORTANCE_SAMPLE_SIZE
                    ),

                posterior_sample_size=
                    np.asarray(
                        GLASS_POSTERIOR_SAMPLE_SIZE
                    ),

                learning_rate=
                    np.asarray(
                        GLASS_LEARNING_RATE
                    ),

                random_seed=
                    np.asarray(
                        GLASS_SEED
                    ),
            )


        temp.replace(
            stage1_path
        )


        print(
            f"    Stage 1: "
            f"{stage1_seconds/60:.2f} min"
        )


        # IMMEDIATE DRIVE PERSISTENCE

        persist_glass_file_strict(
            fold,
            "stage1_no_shrinkage.npz"
        )


        del glass1

        gc.collect()

        tf.keras.backend.clear_session()


    # --------------------------------------------------------
    # Frozen Stage 2 shrinkage
    # --------------------------------------------------------

    cutoff = float(
        np.median(
            np.abs(
                beta_stage1
            )
        )
    )


    shrinkage = float(
        0.5
        *
        cutoff
    )


    print(
        "    median |beta|:",
        cutoff
    )


    print(
        "  GLASS Stage 2: TRAINING..."
    )


    print(
        "    shrinkage:",
        shrinkage
    )


    tf.keras.utils.set_random_seed(
        GLASS_SEED
    )


    glass2 = Glass(
        shrinkage_factor=
            shrinkage,

        dtype=
            tf.float32
    )


    glass2.process_data(
        X_train,
        y_train
    )


    t0 = time.perf_counter()


    with tf.device(
        "/CPU:0"
    ):

        glass2.mfvb(

            num_steps=
                GLASS_NUM_STEPS,

            sample_size=
                GLASS_SAMPLE_SIZE,

            importance_sample_size=
                GLASS_IMPORTANCE_SAMPLE_SIZE,

            learning_rate=
                GLASS_LEARNING_RATE,

            seed=
                GLASS_SEED,
        )


    stage2_seconds = (
        time.perf_counter()
        -
        t0
    )


    beta_stage2 = np.asarray(
        glass2.betaMat,
        dtype=np.float32
    )


    losses_stage2 = np.asarray(
        glass2.losses,
        dtype=np.float32
    )


    assert beta_stage2.shape == (
        8,
        128
    )


    assert np.isfinite(
        beta_stage2
    ).all()


    temp = (
        glass_dir
        / "stage2_shrinkage.tmp"
    )


    with temp.open(
        "wb"
    ) as f:

        np.savez_compressed(

            f,

            betaMat=
                beta_stage2,

            losses=
                losses_stage2,

            cutoff=
                np.asarray(
                    cutoff
                ),

            shrinkage_factor=
                np.asarray(
                    shrinkage
                ),

            stage1_fitting_seconds=
                np.asarray(
                    stage1_seconds
                ),

            stage2_fitting_seconds=
                np.asarray(
                    stage2_seconds
                ),

            total_fitting_seconds=
                np.asarray(
                    stage1_seconds
                    +
                    stage2_seconds
                ),

            training_characters_1based=
                train_chars,

            training_data_only_for_fit=
                np.asarray(
                    True
                ),

            inner_validation_used_for_fitting=
                np.asarray(
                    False
                ),

            outer_test_used_for_fitting=
                np.asarray(
                    False
                ),

            num_steps_per_stage=
                np.asarray(
                    GLASS_NUM_STEPS
                ),

            sample_size=
                np.asarray(
                    GLASS_SAMPLE_SIZE
                ),

            importance_sample_size=
                np.asarray(
                    GLASS_IMPORTANCE_SAMPLE_SIZE
                ),

            posterior_sample_size=
                np.asarray(
                    GLASS_POSTERIOR_SAMPLE_SIZE
                ),

            learning_rate=
                np.asarray(
                    GLASS_LEARNING_RATE
                ),

            random_seed=
                np.asarray(
                    GLASS_SEED
                ),
        )


    temp.replace(
        stage2_path
    )


    print(
        f"    Stage 2: "
        f"{stage2_seconds/60:.2f} min"
    )


    # IMMEDIATE DRIVE PERSISTENCE

    persist_glass_file_strict(
        fold,
        "stage2_shrinkage.npz"
    )


    del glass2

    gc.collect()

    tf.keras.backend.clear_session()


    return beta_stage2


# ============================================================
# 10. MODEL CONSTRUCTION
# ============================================================

def construct_model(
    model_name,
    beta,
):

    if model_name == (
        "structured_dbnet"
    ):

        return (
            build_structured_dbnet()
        )


    if model_name == (
        "glass_gated_dbnet"
    ):

        return (
            build_gated_hybrid(
                beta
            )
        )


    if model_name == (
        "interaction_dbnet"
    ):

        return (
            build_interaction_dbnet()
        )


    if model_name == (
        "binary_dbnet"
    ):

        return (
            build_binary_dbnet_frozen()
        )


    raise ValueError(
        model_name
    )


# ============================================================
# 11. PROGRESS CALLBACK
# ============================================================

class ProgressPrinter(
    tf.keras.callbacks.Callback
):

    def __init__(
        self,
        monitor,
    ):

        super().__init__()

        self.monitor = (
            monitor
        )

        self.start = None


    def on_train_begin(
        self,
        logs=None
    ):

        self.start = (
            time.perf_counter()
        )


    def on_epoch_end(
        self,
        epoch,
        logs=None
    ):

        epoch_number = (
            epoch + 1
        )

        logs = logs or {}


        if (
            epoch_number == 1
            or
            epoch_number % 25 == 0
        ):

            elapsed = (
                time.perf_counter()
                -
                self.start
            ) / 60


            value = logs.get(
                self.monitor
            )


            if value is None:

                print(
                    f"    epoch "
                    f"{epoch_number:4d} | "
                    f"{elapsed:6.1f} min"
                )

            else:

                print(
                    f"    epoch "
                    f"{epoch_number:4d} | "
                    f"{elapsed:6.1f} min | "
                    f"{self.monitor}="
                    f"{float(value):.6f}"
                )


# ============================================================
# 12. TRAIN ONE MODEL
# ============================================================

def train_one_model(
    fold,
    seed,
    model_name,
    beta,
    fold_data,
    fisher_train,
    fisher_val,
    X_binary_train,
    y_binary_train,
    X_binary_val,
    y_binary_val,
):

    run_dir = (
        LOCAL_A02_ROOT
        / f"outer_fold_{fold}"
        / f"seed_{seed}"
        / model_name
    )


    # --------------------------------------------------------
    # Restart-safe reuse
    # --------------------------------------------------------

    if model_complete(
        fold,
        seed,
        model_name
    ):

        print(
            f"  REUSE | "
            f"{model_name} | "
            f"seed {seed}"
        )


        # Ensure persistent Drive copy is valid

        persist_model_strict(
            fold,
            seed,
            model_name
        )


        return


    # Remove incomplete local attempt only

    if run_dir.exists():

        shutil.rmtree(
            run_dir
        )


    run_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    tf.keras.backend.clear_session()

    gc.collect()


    tf.keras.utils.set_random_seed(
        seed
    )


    model = construct_model(
        model_name,
        beta
    )


    assert (
        model.count_params()
        ==
        EXPECTED_PARAMS[
            model_name
        ]
    )


    checkpoint = (
        run_dir
        / "best.weights.h5"
    )


    if model_name == (
        "binary_dbnet"
    ):

        monitor = (
            "val_target_auc"
        )

        mode = "max"

    else:

        monitor = (
            "val_loss"
        )

        mode = "min"


    callbacks = [

        tf.keras.callbacks
        .ModelCheckpoint(

            filepath=
                str(
                    checkpoint
                ),

            monitor=
                monitor,

            mode=
                mode,

            save_best_only=
                True,

            save_weights_only=
                True,

            verbose=
                0,
        ),


        tf.keras.callbacks
        .EarlyStopping(

            monitor=
                monitor,

            mode=
                mode,

            patience=
                PATIENCE,

            min_delta=
                MIN_DELTA,

            restore_best_weights=
                False,

            verbose=
                0,
        ),


        ProgressPrinter(
            monitor
        ),
    ]


    print(
        f"\n  TRAINING "
        f"{model_name} "
        f"seed {seed}"
    )


    t0 = time.perf_counter()


    if model_name == (
        "interaction_dbnet"
    ):

        history = model.fit(

            [
                fold_data[
                    "X_train"
                ],
                fisher_train,
            ],

            fold_data[
                "y_train"
            ],

            validation_data=(

                [
                    fold_data[
                        "X_val"
                    ],
                    fisher_val,
                ],

                fold_data[
                    "y_val"
                ],
            ),

            epochs=
                MAX_EPOCHS,

            batch_size=
                GROUP_BATCH_SIZE,

            shuffle=
                True,

            callbacks=
                callbacks,

            verbose=
                0,
        )


    elif model_name == (
        "binary_dbnet"
    ):

        history = model.fit(

            X_binary_train,

            y_binary_train,

            validation_data=(

                X_binary_val,

                y_binary_val,
            ),

            epochs=
                MAX_EPOCHS,

            batch_size=
                BINARY_BATCH_SIZE,

            shuffle=
                True,

            callbacks=
                callbacks,

            verbose=
                0,
        )


    else:

        history = model.fit(

            fold_data[
                "X_train"
            ],

            fold_data[
                "y_train"
            ],

            validation_data=(

                fold_data[
                    "X_val"
                ],

                fold_data[
                    "y_val"
                ],
            ),

            epochs=
                MAX_EPOCHS,

            batch_size=
                GROUP_BATCH_SIZE,

            shuffle=
                True,

            callbacks=
                callbacks,

            verbose=
                0,
        )


    runtime_seconds = (
        time.perf_counter()
        -
        t0
    )


    assert checkpoint.is_file(), (
        f"Checkpoint missing: "
        f"{checkpoint}"
    )


    # Verify checkpoint can load

    model.load_weights(
        checkpoint
    )


    values = np.asarray(
        history.history[
            monitor
        ],
        dtype=np.float64
    )


    if mode == "min":

        best_index = int(
            np.argmin(
                values
            )
        )

    else:

        best_index = int(
            np.argmax(
                values
            )
        )


    best_epoch = (
        best_index + 1
    )


    best_value = float(
        values[
            best_index
        ]
    )


    # --------------------------------------------------------
    # Save training history locally
    # --------------------------------------------------------

    pd.DataFrame(
        history.history
    ).to_csv(
        run_dir
        / "training_history.csv",
        index=False
    )


    # --------------------------------------------------------
    # Save inner-validation predictions only
    # --------------------------------------------------------

    if model_name == (
        "interaction_dbnet"
    ):

        val_prob = model.predict(

            [
                fold_data[
                    "X_val"
                ],
                fisher_val,
            ],

            batch_size=
                GROUP_BATCH_SIZE,

            verbose=
                0,
        )


    elif model_name == (
        "binary_dbnet"
    ):

        val_prob = model.predict(

            X_binary_val,

            batch_size=
                BINARY_BATCH_SIZE,

            verbose=
                0,
        )


    else:

        val_prob = model.predict(

            fold_data[
                "X_val"
            ],

            batch_size=
                GROUP_BATCH_SIZE,

            verbose=
                0,
        )


    np.savez_compressed(

        run_dir
        / "validation_predictions.npz",

        probabilities=
            np.asarray(
                val_prob,
                dtype=np.float32
            ),

        seed=
            np.asarray(
                seed,
                dtype=np.int64
            ),

        best_epoch=
            np.asarray(
                best_epoch,
                dtype=np.int64
            ),

        runtime_seconds=
            np.asarray(
                runtime_seconds
            ),
    )


    summary = {

        "status":
            "completed",

        "subject":
            SUBJECT,

        "outer_fold":
            int(fold),

        "seed":
            int(seed),

        "model":
            model_name,

        "parameter_count":
            int(
                model.count_params()
            ),

        "monitor":
            monitor,

        "monitor_mode":
            mode,

        "best_epoch":
            int(
                best_epoch
            ),

        "best_monitor_value":
            best_value,

        "runtime_seconds":
            float(
                runtime_seconds
            ),

        "learning_rate":
            LEARNING_RATE,

        "maximum_epochs":
            MAX_EPOCHS,

        "patience":
            PATIENCE,

        "min_delta":
            MIN_DELTA,

        "training_characters_1based":
            fold_data[
                "train_chars"
            ],

        "inner_validation_characters_1based":
            fold_data[
                "val_chars"
            ],

        "outer_characters_1based":
            fold_data[
                "outer_chars"
            ],

        "outer_test_used_for_training":
            False,

        "outer_test_used_for_model_selection":
            False,

        "outer_performance_evaluated":
            False,

        "completed_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }


    atomic_json_save(
        run_dir
        / "training_summary.json",
        summary
    )


    print(
        f"  COMPLETE | "
        f"{model_name} | "
        f"seed {seed} | "
        f"{runtime_seconds/60:.2f} min | "
        f"best epoch {best_epoch} | "
        f"{monitor}="
        f"{best_value:.6f}"
    )


    # ========================================================
    # CRITICAL:
    # IMMEDIATELY COPY COMPLETED MODEL TO DRIVE.
    # ========================================================

    persist_model_strict(
        fold,
        seed,
        model_name
    )


    del model
    del history
    del val_prob

    gc.collect()

    tf.keras.backend.clear_session()


# ============================================================
# 13. CHECKPOINT LOAD AUDIT
# ============================================================

def audit_fold_checkpoints(
    fold,
    beta,
):

    print(
        "\nCHECKPOINT LOAD AUDIT"
    )


    count = 0


    for seed in SEEDS:

        for model_name in [

            "structured_dbnet",

            "glass_gated_dbnet",

            "interaction_dbnet",

            "binary_dbnet",
        ]:

            assert model_complete(
                fold,
                seed,
                model_name
            )


            tf.keras.backend.clear_session()

            gc.collect()


            tf.keras.utils.set_random_seed(
                seed
            )


            model = construct_model(
                model_name,
                beta
            )


            path = (
                LOCAL_A02_ROOT
                / f"outer_fold_{fold}"
                / f"seed_{seed}"
                / model_name
                / "best.weights.h5"
            )


            model.load_weights(
                path
            )


            for variable in (
                model.weights
            ):

                assert np.isfinite(
                    variable.numpy()
                ).all()


            print(
                f"PASS | "
                f"seed {seed} | "
                f"{model_name}"
            )


            count += 1

            del model


    assert count == 12


    tf.keras.backend.clear_session()

    gc.collect()


# ============================================================
# 14. TRAIN ONE FOLD
# ============================================================

def run_one_fold(
    fold,
):

    fold_start = (
        time.perf_counter()
    )


    print(
        "\n"
        +
        "=" * 100
    )

    print(
        f"A02 OUTER FOLD {fold}"
    )

    print(
        "=" * 100
    )


    fold_data = (
        prepare_fold_data(
            fold
        )
    )


    print(
        "Train:",
        fold_data[
            "train_chars"
        ]
    )

    print(
        "Inner val:",
        fold_data[
            "val_chars"
        ]
    )

    print(
        "Outer indices reserved:",
        fold_data[
            "outer_chars"
        ]
    )

    print(
        "Outer EEG/labels accessed: NO"
    )


    # --------------------------------------------------------
    # GLASS
    # --------------------------------------------------------

    beta = run_glass(
        fold,
        fold_data
    )


    print(
        "  Fold-local GLASS: PASS"
    )

    print(
        "  beta:",
        beta.shape
    )


    # --------------------------------------------------------
    # Fisher features
    # --------------------------------------------------------

    print(
        "\nComputing training Fisher-z..."
    )


    fisher_train = (
        compute_fisher_features(
            fold_data[
                "X_train"
            ]
        )
    )


    print(
        "Computing validation Fisher-z..."
    )


    fisher_val = (
        compute_fisher_features(
            fold_data[
                "X_val"
            ]
        )
    )


    assert fisher_train.shape == (
        500,
        6,
        28
    )


    assert fisher_val.shape == (
        100,
        6,
        28
    )


    # --------------------------------------------------------
    # Binary views
    # --------------------------------------------------------

    X_binary_train = (
        fold_data[
            "X_train"
        ]
        .reshape(
            3000,
            8,
            128,
            1
        )
    )


    binary_train_labels = (
        fold_data[
            "y_train"
        ]
        .reshape(-1)
        .astype(
            np.int64
        )
    )


    y_binary_train = (
        tf.keras.utils
        .to_categorical(
            binary_train_labels,
            num_classes=2
        )
        .astype(
            np.float32
        )
    )


    X_binary_val = (
        fold_data[
            "X_val"
        ]
        .reshape(
            600,
            8,
            128,
            1
        )
    )


    binary_val_labels = (
        fold_data[
            "y_val"
        ]
        .reshape(-1)
        .astype(
            np.int64
        )
    )


    y_binary_val = (
        tf.keras.utils
        .to_categorical(
            binary_val_labels,
            num_classes=2
        )
        .astype(
            np.float32
        )
    )


    print(
        "\nBinary train:",
        X_binary_train.shape
    )


    print(
        "Targets/non-targets:",
        int(
            binary_train_labels.sum()
        ),
        int(
            len(binary_train_labels)
            -
            binary_train_labels.sum()
        )
    )


    # --------------------------------------------------------
    # Structured family
    # --------------------------------------------------------

    for seed in SEEDS:

        print(
            "\n"
            +
            "-" * 80
        )

        print(
            f"STRUCTURED FAMILY — "
            f"SEED {seed}"
        )

        print(
            "-" * 80
        )


        for model_name in (
            STRUCTURED_ORDER[
                seed
            ]
        ):

            train_one_model(

                fold,
                seed,
                model_name,
                beta,
                fold_data,
                fisher_train,
                fisher_val,
                X_binary_train,
                y_binary_train,
                X_binary_val,
                y_binary_val,
            )


    # --------------------------------------------------------
    # Binary family
    # --------------------------------------------------------

    for seed in SEEDS:

        print(
            "\n"
            +
            "-" * 80
        )

        print(
            f"BINARY DBNET — "
            f"SEED {seed}"
        )

        print(
            "-" * 80
        )


        train_one_model(

            fold,
            seed,
            "binary_dbnet",
            beta,
            fold_data,
            fisher_train,
            fisher_val,
            X_binary_train,
            y_binary_train,
            X_binary_val,
            y_binary_val,
        )


    # --------------------------------------------------------
    # Hard checkpoint audit
    # --------------------------------------------------------

    audit_fold_checkpoints(
        fold,
        beta
    )


    wall_seconds = (
        time.perf_counter()
        -
        fold_start
    )


    # --------------------------------------------------------
    # Final fold marker
    # --------------------------------------------------------

    marker = (
        LOCAL_A02_ROOT
        / f"outer_fold_{fold}"
        / "fold_training_complete.json"
    )


    atomic_json_save(

        marker,

        {

            "status":
                "training_complete",

            "subject":
                SUBJECT,

            "outer_fold":
                int(fold),

            "training_characters_1based":
                fold_data[
                    "train_chars"
                ],

            "inner_validation_characters_1based":
                fold_data[
                    "val_chars"
                ],

            "outer_characters_indices_only_1based":
                fold_data[
                    "outer_chars"
                ],

            "glass_fit_complete":
                True,

            "neural_checkpoints_complete":
                12,

            "fold_wall_seconds":
                float(
                    wall_seconds
                ),

            "outer_test_eeg_accessed":
                False,

            "outer_performance_evaluated":
                False,

            "completed_utc":
                datetime.now(
                    timezone.utc
                ).isoformat(),
        },
    )


    # Persist marker to Drive

    persist_fold_marker_strict(
        fold
    )


    print(
        "\n"
        +
        "=" * 100
    )

    print(
        f"A02 FOLD {fold} — "
        "TRAINING COMPLETE + "
        "DRIVE VERIFIED"
    )

    print(
        "=" * 100
    )

    print(
        "All 12 neural checkpoints: PASS"
    )

    print(
        f"Fold wall time: "
        f"{wall_seconds/60:.2f} min"
    )

    print(
        "Outer EEG/labels accessed: NO"
    )

    print(
        "Outer performance evaluated: NO"
    )


    del fold_data
    del fisher_train
    del fisher_val
    del X_binary_train
    del X_binary_val
    del y_binary_train
    del y_binary_val

    gc.collect()

    tf.keras.backend.clear_session()


# ============================================================
# 15. MAIN FOLDS 3–7 QUEUE
# ============================================================

print(
    "\n"
    +
    "=" * 100
)

print(
    "STARTING A02 FAST "
    "FOLDS 3–7 RESUME QUEUE"
)

print(
    "=" * 100
)

print(
    "Training location:"
)

print(
    LOCAL_A02_ROOT
)

print(
    "\nPersistent Drive location:"
)

print(
    DRIVE_A02_ROOT
)

print(
    "\nFolds:",
    FOLDS_TO_RUN
)

print(
    "Seeds:",
    SEEDS
)

print(
    "Outer EEG access: DISABLED"
)

print(
    "Outer evaluation: DISABLED"
)


overall_start = (
    time.perf_counter()
)


for fold in FOLDS_TO_RUN:

    local_marker = (
        LOCAL_A02_ROOT
        / f"outer_fold_{fold}"
        / "fold_training_complete.json"
    )


    drive_marker = (
        DRIVE_A02_ROOT
        / f"outer_fold_{fold}"
        / "fold_training_complete.json"
    )


    if (
        local_marker.is_file()
        and
        drive_marker.is_file()
    ):

        print(
            f"\nA02 Fold {fold}: "
            "ALREADY COMPLETE + "
            "PERSISTED — SKIPPING"
        )

        continue


    run_one_fold(
        fold
    )


# ============================================================
# 16. FINAL DRIVE AUDIT
# ============================================================

print(
    "\n"
    +
    "=" * 100
)

print(
    "A02 FOLDS 3–7 FINAL DRIVE AUDIT"
)

print(
    "=" * 100
)


all_complete = True


for fold in FOLDS_TO_RUN:

    fold_dir = (
        DRIVE_A02_ROOT
        / f"outer_fold_{fold}"
    )


    stage1 = (
        fold_dir
        / "glass"
        / "stage1_no_shrinkage.npz"
    )


    stage2 = (
        fold_dir
        / "glass"
        / "stage2_shrinkage.npz"
    )


    count = 0


    for seed in SEEDS:

        for model_name in [

            "structured_dbnet",

            "glass_gated_dbnet",

            "interaction_dbnet",

            "binary_dbnet",
        ]:

            run_dir = (
                fold_dir
                / f"seed_{seed}"
                / model_name
            )


            if (
                (
                    run_dir
                    / "best.weights.h5"
                ).is_file()

                and

                (
                    run_dir
                    / "training_summary.json"
                ).is_file()
            ):

                count += 1


    marker = (
        fold_dir
        / "fold_training_complete.json"
    )


    fold_ok = (

        stage1.is_file()

        and

        stage2.is_file()

        and

        count == 12

        and

        marker.is_file()
    )


    print(
        f"Fold {fold}: "
        f"GLASS="
        f"{'PASS' if stage2.is_file() else 'FAIL'} | "
        f"checkpoints={count}/12 | "
        f"marker="
        f"{'PASS' if marker.is_file() else 'FAIL'}"
    )


    all_complete = (
        all_complete
        and
        fold_ok
    )


overall_seconds = (
    time.perf_counter()
    -
    overall_start
)


assert all_complete, (
    "Not all A02 folds are complete "
    "and persistent."
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "A02 FOLDS 3–7 TRAINING: COMPLETE"
)

print(
    "ALL CHECKPOINTS PERSISTED TO DRIVE: PASS"
)

print(
    "=" * 100
)

print(
    f"\nThis-session wall time: "
    f"{overall_seconds/3600:.2f} h"
)

print(
    "\nOuter EEG accessed: NO"
)

print(
    "Outer performance evaluated: NO"
)

print(
    "\nSAFE TO RUN LOCKED A02 "
    "OUTER EVALUATION."
)

In [ ]:
give